In [1]:
import os
from uuid import uuid4
from datetime import datetime, timezone
import json as _json
import hashlib

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

MINIO_STAGING_BUCKET = os.environ.get("MINIO_STAGING_BUCKET", "staging")
MINIO_ACCESS_KEY = os.environ.get("MINIO_ROOT_USER", "minio")
MINIO_SECRET_KEY = os.environ.get("MINIO_ROOT_PASSWORD", "")
MINIO_ENDPOINT = os.environ.get("MINIO_ENDPOINT", f"http://urbangreen-minio:{os.environ.get('MINIO_API_PORT', '9000')}")

RAW_POSTGRES_PREFIX = f"s3a://{MINIO_STAGING_BUCKET}/raw/postgres"
RAW_KAFKA_PREFIX = f"s3a://{MINIO_STAGING_BUCKET}/raw/kafka"

CLICKHOUSE_HOST = os.environ.get("CLICKHOUSE_HOST", "urbangreen-clickhouse")
CLICKHOUSE_DB = os.environ.get("CLICKHOUSE_DB", "urbangreen_dw")
CLICKHOUSE_USER = os.environ.get("CLICKHOUSE_USER", "urbangreen")
CLICKHOUSE_PASSWORD = os.environ.get("CLICKHOUSE_PASSWORD", "")
CLICKHOUSE_HTTP_PORT = "8123"

PREMIUM_QUALITY_CODES = ["A"]
ENERGY_SENSOR_TYPE = "Energy Usage"

DRIVER = "com.clickhouse.jdbc.ClickHouseDriver"
JDBC_URL = f"jdbc:clickhouse://{CLICKHOUSE_HOST}:{CLICKHOUSE_HTTP_PORT}/{CLICKHOUSE_DB}"

FAR_FUTURE = "2099-12-31 23:59:59"
INITIAL_VALID_FROM = "1970-01-01 00:00:00"
CHANGE_HASH = "_change_hash"
STATE_TABLE = "warehouse_load_state"
UNKNOWN_FARM_KEY = 0

In [2]:
spark = (
    SparkSession.builder.appName("warehouse_loaders")
    .master("local[*]")
    .config("spark.jars.packages", "com.clickhouse:clickhouse-jdbc:0.9.8,org.apache.hadoop:hadoop-aws:3.4.1")
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/jovyan/.ivy2.5.2/cache
The jars for the packages stored in: /home/jovyan/.ivy2.5.2/jars
com.clickhouse#clickhouse-jdbc added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-232e89ba-439d-4eb3-ae00-f5307d809018;1.0
	confs: [default]
	found com.clickhouse#clickhouse-jdbc;0.9.8 in central
	found com.clickhouse#clickhouse-client;0.9.8 in central
	found org.apache.commons#commons-compress;1.28.0 in central
	found commons-codec#commons-codec;1.19.0 in central
	found commons-io#commons-io;2.20.0 in central
	found org.apache.commons#commons-lang3;3.20.0 in central
	found com.clickhouse#clickhouse-data;0.9.8 in central
	found com.clickhouse#clickhouse-http-client;0.9.8 in central
	found org.apache.httpcomponents.core5#httpcore5;5.3

In [3]:
def read_raw_postgres(table):
    path = f"{RAW_POSTGRES_PREFIX}/{table}/*/{table}.parquet"
    try:
        return spark.read.parquet(path)
    except Exception:
        return None


def read_raw_postgres_partitioned(table):
    path = f"{RAW_POSTGRES_PREFIX}/{table}/*/*/*.parquet"
    try:
        return spark.read.parquet(path)
    except Exception:
        return None


def read_raw_kafka(topic):
    path = f"{RAW_KAFKA_PREFIX}/{topic}"
    try:
        return spark.read.parquet(path)
    except Exception:
        return None

In [4]:
def _properties():
    return {"user": CLICKHOUSE_USER, "password": CLICKHOUSE_PASSWORD, "driver": DRIVER}


def read_query(query):
    return spark.read.jdbc(url=JDBC_URL, table=f"({query}) AS subquery", properties=_properties())


def table_exists(table):
    query = f"SELECT count() AS matches FROM system.tables WHERE database = '{CLICKHOUSE_DB}' AND name = '{table}'"
    return read_query(query).collect()[0]["matches"] > 0


def write_table(df, table, mode="append"):
    if not table_exists(table):
        raise RuntimeError(f"{CLICKHOUSE_DB}.{table} does not exist")
    df = df.persist()
    try:
        count = df.count()
        df.write.jdbc(url=JDBC_URL, table=table, mode=mode, properties=_properties())
        print(f"wrote {count} row(s) -> {CLICKHOUSE_DB}.{table}")
        return count
    finally:
        df.unpersist()

In [5]:
def epoch_to_ts(column):
    return F.col(column).cast("long").cast("timestamp")


def latest_per_key(df, key, order_by="updated_at"):
    window = Window.partitionBy(key).orderBy(F.col(order_by).desc())
    return df.withColumn("_row", F.row_number().over(window)).filter(F.col("_row") == 1).drop("_row")


def change_hash(columns):
    parts = [F.coalesce(F.col(c).cast("string"), F.lit("")) for c in columns]
    return F.sha2(F.concat_ws("||", *parts), 256)


def apply_scd2(incoming, table, natural_key, tracked_columns, surrogate_column):
    version = int(datetime.now(timezone.utc).timestamp() * 1000)
    current = read_query(f"SELECT * FROM {table} FINAL WHERE is_current = 1").drop(surrogate_column, "_version")
    incoming = incoming.withColumn(CHANGE_HASH, change_hash(tracked_columns))
    current = current.withColumn(CHANGE_HASH, change_hash(tracked_columns))
    paired = incoming.alias("i").join(current.alias("c"), F.col(f"i.{natural_key}") == F.col(f"c.{natural_key}"), "left")
    is_new = F.col(f"c.{natural_key}").isNull()
    is_changed = F.col(f"i.{CHANGE_HASH}") != F.col(f"c.{CHANGE_HASH}")
    is_newer = F.col("i.valid_from") > F.col("c.valid_from")
    business_columns = [c for c in incoming.columns if c != CHANGE_HASH]

    def opened_column(column):
        if column == "valid_from":
            return F.when(is_new, F.lit(INITIAL_VALID_FROM).cast("timestamp")).otherwise(F.col("i.valid_from")).alias("valid_from")
        return F.col(f"i.{column}").alias(column)

    opened = (
        paired.filter(is_new | (is_changed & is_newer))
        .select([opened_column(c) for c in business_columns])
        .withColumn("valid_to", F.lit(FAR_FUTURE).cast("timestamp"))
        .withColumn("is_current", F.lit(1).cast("tinyint"))
        .withColumn("_version", F.lit(version).cast("long"))
    )
    closing_columns = [c for c in current.columns if c != CHANGE_HASH]
    closed = (
        paired.filter(is_changed & is_newer)
        .select([F.col(f"c.{c}").alias(c) for c in closing_columns] + [F.col("i.valid_from").alias("_new_valid_from")])
        .withColumn("valid_to", F.col("_new_valid_from"))
        .withColumn("is_current", F.lit(0).cast("tinyint"))
        .withColumn("_version", F.lit(version).cast("long"))
        .drop("_new_valid_from")
    )
    closed_count = write_table(closed, table) if not closed.isEmpty() else 0
    opened_count = write_table(opened, table) if not opened.isEmpty() else 0
    print(f"{table}: opened {opened_count}, closed {closed_count}")
    return opened_count, closed_count


def date_key(column):
    return F.date_format(F.col(column), "yyyyMMdd").cast("int")


def time_key(column):
    return (F.hour(F.col(column)) * 100 + F.minute(F.col(column))).cast("int")


def as_of_version(facts, dim, natural_key, event_ts, attributes):
    matched = facts.alias("f").join(
        dim.alias("d"),
        (F.col(f"f.{natural_key}") == F.col(f"d.{natural_key}"))
        & (F.col(f"f.{event_ts}") >= F.col("d.valid_from"))
        & (F.col(f"f.{event_ts}") < F.col("d.valid_to")),
        "left",
    )
    selected = [F.col(f"f.{c}").alias(c) for c in facts.columns]
    selected += [F.coalesce(F.col(f"d.{source}"), F.lit(fallback)).alias(alias) for source, alias, fallback in attributes]
    return matched.select(selected)


def read_cursor(job_name):
    query = f"SELECT cursor_json FROM {STATE_TABLE} FINAL WHERE job_name = '{job_name}'"
    rows = read_query(query).collect()
    if not rows:
        return None
    return _json.loads(rows[0]["cursor_json"])


def write_cursor(job_name, cursor):
    now = datetime.now(timezone.utc)
    schema = "job_name string, cursor_json string, last_success_at timestamp, run_key string, _version long"
    row = [(job_name, _json.dumps(cursor, sort_keys=True), now, str(uuid4()), int(now.timestamp() * 1000))]
    write_table(spark.createDataFrame(row, schema=schema), STATE_TABLE)

In [6]:
raw = read_raw_postgres("roles")
current = latest_per_key(raw, "id")
target = current.select(
    F.col("id").cast("long").alias("role_id"),
    F.col("name").cast("string").alias("name"),
    F.coalesce(F.col("description"), F.lit("")).cast("string").alias("description"),
)
write_table(target, "dim_role")

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


wrote 3 row(s) -> urbangreen_dw.dim_role


3

In [7]:
raw = read_raw_postgres("quality_grades")
current = latest_per_key(raw, "id")
target = current.select(
    F.col("id").cast("long").alias("quality_grade_id"),
    F.col("code").cast("string").alias("code"),
    F.col("name").cast("string").alias("name"),
    F.coalesce(F.col("description"), F.lit("")).cast("string").alias("description"),
    F.col("code").isin(PREMIUM_QUALITY_CODES).cast("tinyint").alias("is_premium"),
)
write_table(target, "dim_quality_grade")

wrote 5 row(s) -> urbangreen_dw.dim_quality_grade


5

In [8]:
raw_crops = read_raw_postgres("crops")
crops = latest_per_key(raw_crops, "id").select(
    F.col("id").cast("long").alias("crop_id"),
    F.col("name").cast("string").alias("name"),
    F.coalesce(F.col("description"), F.lit("")).cast("string").alias("description"),
    F.col("category_id").cast("long").alias("crop_category_id"),
)
raw_categories = read_raw_postgres("crop_categories")
categories = latest_per_key(raw_categories, "id").select(
    F.col("id").cast("long").alias("crop_category_id"),
    F.col("name").cast("string").alias("category_name"),
)
enriched = crops.join(categories, on="crop_category_id", how="left")
target = enriched.select(
    "crop_id", "name", "description", "crop_category_id",
    F.coalesce(F.col("category_name"), F.lit("")).alias("category_name"),
)
write_table(target, "dim_crop")

wrote 19 row(s) -> urbangreen_dw.dim_crop


19

In [9]:
raw = read_raw_postgres("users")
current = latest_per_key(raw, "id")
target = current.select(
    F.col("id").cast("long").alias("user_id"),
    F.col("email").cast("string").alias("email"),
    F.col("full_name").cast("string").alias("full_name"),
    F.col("is_active").cast("tinyint").alias("is_active"),
    F.col("created_at").cast("timestamp").alias("created_at"),
)
write_table(target, "dim_user")

wrote 1 row(s) -> urbangreen_dw.dim_user


1

In [10]:
raw = read_raw_postgres("sensor_types")
incoming = latest_per_key(raw, "id").select(
    F.col("id").cast("long").alias("sensor_type_id"),
    F.col("name").cast("string").alias("name"),
    F.col("unit").cast("string").alias("unit"),
    F.coalesce(F.col("description"), F.lit("")).cast("string").alias("description"),
    F.coalesce(F.col("optimal_min"), F.lit(0)).cast("double").alias("optimal_min"),
    F.coalesce(F.col("optimal_max"), F.lit(0)).cast("double").alias("optimal_max"),
    epoch_to_ts("updated_at").alias("valid_from"),
)
apply_scd2(incoming, "dim_sensor_type", "sensor_type_id", ["name", "unit", "description", "optimal_min", "optimal_max"], "sensor_type_key")

dim_sensor_type: opened 0, closed 0


(0, 0)

In [11]:
def lookup_names(table, id_alias, name_alias):
    raw = read_raw_postgres(table)
    if raw is None:
        return None
    return latest_per_key(raw, "id").select(
        F.col("id").cast("long").alias(id_alias),
        F.col("name").cast("string").alias(name_alias),
    )


def attach(df, lookup, join_column, name_column):
    if lookup is None:
        return df.withColumn(name_column, F.lit(""))
    joined = df.join(lookup, on=join_column, how="left")
    return joined.withColumn(name_column, F.coalesce(F.col(name_column), F.lit("")))


raw = read_raw_postgres("farms")
farms = latest_per_key(raw, "id").select(
    F.col("id").cast("long").alias("farm_id"),
    F.col("name").cast("string").alias("name"),
    F.coalesce(F.col("city"), F.lit("")).cast("string").alias("city"),
    F.coalesce(F.col("size_m2"), F.lit(0)).cast("decimal(10,3)").alias("size_m2"),
    F.coalesce(F.col("growing_beds_count"), F.lit(0)).cast("int").alias("growing_beds_count"),
    F.col("status").cast("string").alias("status"),
    F.col("infrastructure_type_id").cast("long").alias("infrastructure_type_id"),
    F.col("growing_system_type_id").cast("long").alias("growing_system_type_id"),
    epoch_to_ts("updated_at").alias("valid_from"),
)
farms = attach(farms, lookup_names("farm_infrastructure_types", "infrastructure_type_id", "infrastructure_type_name"), "infrastructure_type_id", "infrastructure_type_name")
farms = attach(farms, lookup_names("growing_system_types", "growing_system_type_id", "growing_system_type_name"), "growing_system_type_id", "growing_system_type_name")
incoming = farms.select(
    "farm_id", "name", "city", "size_m2", "growing_beds_count", "status",
    "infrastructure_type_id", "infrastructure_type_name",
    "growing_system_type_id", "growing_system_type_name", "valid_from",
)
apply_scd2(incoming, "dim_farm", "farm_id",
           ["name", "city", "size_m2", "growing_beds_count", "status", "infrastructure_type_id", "infrastructure_type_name", "growing_system_type_id", "growing_system_type_name"],
           "farm_key")

dim_farm: opened 0, closed 0


(0, 0)

In [12]:
raw = read_raw_postgres("sensors")
incoming = latest_per_key(raw, "id").select(
    F.col("id").cast("long").alias("sensor_id"),
    F.col("farm_id").cast("long").alias("farm_key"),
    F.col("sensor_type_id").cast("long").alias("sensor_type_key"),
    F.col("serial_number").cast("string").alias("serial_number"),
    F.col("status").cast("string").alias("status"),
    epoch_to_ts("installed_at").alias("installed_at"),
    epoch_to_ts("updated_at").alias("valid_from"),
)
apply_scd2(incoming, "dim_sensor", "sensor_id", ["farm_key", "sensor_type_key", "serial_number", "status", "installed_at"], "sensor_key")

dim_sensor: opened 0, closed 0


(0, 0)

In [13]:
def current_names(query, key_column, name_column):
    return read_query(query).select(
        F.col(key_column).cast("long").alias(key_column),
        F.col(name_column).cast("string").alias(name_column),
    )


raw = read_raw_postgres("user_roles")
assignments = latest_per_key(raw, "id").select(
    F.col("id").cast("long").alias("user_role_id"),
    F.col("user_id").cast("long").alias("user_id"),
    F.col("role_id").cast("long").alias("role_id"),
    F.coalesce(F.col("farm_id"), F.lit(0)).cast("long").alias("farm_id"),
    epoch_to_ts("updated_at").alias("valid_from"),
).withColumn("farm_key", F.col("farm_id"))
users = current_names("SELECT user_id, full_name FROM dim_user FINAL", "user_id", "full_name").withColumnRenamed("full_name", "user_full_name")
roles = current_names("SELECT role_id, name FROM dim_role FINAL", "role_id", "name").withColumnRenamed("name", "role_name")
farms = current_names("SELECT farm_id, name FROM dim_farm FINAL WHERE is_current = 1", "farm_id", "name").withColumnRenamed("name", "farm_name")
incoming = (
    assignments.join(users, on="user_id", how="left")
    .join(roles, on="role_id", how="left")
    .join(farms, on="farm_id", how="left")
    .select(
        "user_role_id", "user_id", "role_id", "farm_key", "farm_id",
        F.coalesce(F.col("user_full_name"), F.lit("")).alias("user_full_name"),
        F.coalesce(F.col("role_name"), F.lit("")).alias("role_name"),
        F.coalesce(F.col("farm_name"), F.lit("")).alias("farm_name"),
        "valid_from",
    )
)
apply_scd2(incoming, "dim_user_farm_role", "user_role_id", ["user_id", "role_id", "farm_id", "farm_key"], "user_role_key")

dim_user_farm_role: opened 0, closed 0


(0, 0)

In [14]:
JOB_NAME = "load_fact_harvests"
raw = read_raw_postgres_partitioned("harvests")
cursor = read_cursor(JOB_NAME) or {}
last_updated_at = cursor.get("updated_at", 0)
changed = raw.filter(F.col("updated_at") > F.lit(last_updated_at))
current = latest_per_key(changed, "id")
harvests = current.select(
    F.col("id").cast("long").alias("harvest_id"),
    F.col("farm_id").cast("long").alias("farm_id"),
    F.col("crop_id").cast("long").alias("crop_id"),
    F.col("quality_grade_id").cast("long").alias("quality_grade_id"),
    F.col("weight_kg").cast("decimal(10,3)").alias("weight_kg"),
    epoch_to_ts("created_at").alias("harvested_at"),
    F.col("updated_at").cast("long").alias("_source_updated_at"),
)
harvests = (
    harvests.withColumn("harvest_date", F.to_date(F.col("harvested_at")))
    .withColumn("date_key", date_key("harvested_at"))
    .withColumn("time_key", time_key("harvested_at"))
    .withColumn("harvest_key", F.xxhash64(F.col("harvest_id")))
)
farm_versions = read_query("SELECT farm_id, farm_key, valid_from, valid_to FROM dim_farm FINAL")
enriched = as_of_version(harvests, farm_versions, "farm_id", "harvested_at", [("farm_key", "farm_key", UNKNOWN_FARM_KEY)]).persist()
target = enriched.select(
    "harvest_key", "harvest_id", "farm_key", "farm_id", "crop_id", "quality_grade_id",
    "date_key", "time_key", "harvested_at", "harvest_date", "weight_kg",
)
write_table(target, "fact_harvests")
high_water = enriched.agg(F.max("_source_updated_at")).collect()[0][0]
if high_water is not None:
    write_cursor(JOB_NAME, {"updated_at": int(high_water)})

wrote 0 row(s) -> urbangreen_dw.fact_harvests


In [15]:
JOB_NAME = "load_fact_sensor_readings"
NO_ANOMALY_MIN = float("-inf")
NO_ANOMALY_MAX = float("inf")
raw = read_raw_kafka("sensor_readings")
cursor = read_cursor(JOB_NAME) or {}
last_date = cursor.get("event_date")
window = raw.filter(F.col("event_date") >= F.lit(last_date)) if last_date else raw
readings = window.select(
    F.col("farm_sensor_id").cast("long").alias("sensor_key"),
    F.col("farm_id").cast("long").alias("farm_id"),
    F.col("sensor_type_id").cast("long").alias("sensor_type_key"),
    F.col("value").cast("double").alias("value"),
    F.col("timestamp").cast("long").cast("timestamp").alias("reading_ts"),
)
readings = (
    readings.withColumn("reading_date", F.to_date(F.col("reading_ts")))
    .withColumn("date_key", date_key("reading_ts"))
    .withColumn("time_key", time_key("reading_ts"))
    .withColumn("reading_key", F.xxhash64(F.col("sensor_key"), F.col("reading_ts")))
)
farm_versions = read_query("SELECT farm_id, farm_key, valid_from, valid_to FROM dim_farm FINAL")
with_farm = as_of_version(readings, farm_versions, "farm_id", "reading_ts", [("farm_key", "farm_key", UNKNOWN_FARM_KEY)])
type_versions = read_query("SELECT sensor_type_id AS sensor_type_key, optimal_min, optimal_max, valid_from, valid_to FROM dim_sensor_type FINAL")
with_thresholds = as_of_version(with_farm, type_versions, "sensor_type_key", "reading_ts", [("optimal_min", "optimal_min", NO_ANOMALY_MIN), ("optimal_max", "optimal_max", NO_ANOMALY_MAX)])
target = with_thresholds.withColumn(
    "is_anomaly", ((F.col("value") < F.col("optimal_min")) | (F.col("value") > F.col("optimal_max"))).cast("tinyint")
).select(
    "reading_key", "farm_key", "farm_id", "sensor_key", "sensor_type_key",
    "date_key", "time_key", "reading_ts", "reading_date", "value", "is_anomaly",
)
write_table(target, "fact_sensor_readings")
high_water = target.agg(F.max("reading_date")).collect()[0][0]
if high_water is not None:
    write_cursor(JOB_NAME, {"event_date": str(high_water)})

wrote 3600 row(s) -> urbangreen_dw.fact_sensor_readings


wrote 1 row(s) -> urbangreen_dw.warehouse_load_state


In [16]:
harvests = read_query("SELECT farm_id, farm_key, harvest_date, date_key, quality_grade_id, weight_kg FROM fact_harvests FINAL")
rollup = harvests.groupBy("farm_id", "harvest_date", "date_key", "quality_grade_id").agg(
    F.max("farm_key").alias("farm_key"),
    F.sum("weight_kg").cast("decimal(18,3)").alias("total_yield_kg"),
    F.count(F.lit(1)).cast("int").alias("harvest_count"),
)
target = rollup.select(
    F.col("harvest_date").alias("metric_date"), "date_key", "farm_key", "farm_id",
    "quality_grade_id", "total_yield_kg", "harvest_count",
)
write_table(target, "fact_daily_farm_quality_metrics")

wrote 228037 row(s) -> urbangreen_dw.fact_daily_farm_quality_metrics


228037

In [17]:
readings = read_query("SELECT farm_id, farm_key, sensor_type_key, reading_date, date_key, value, is_anomaly FROM fact_sensor_readings FINAL")
rollup = readings.groupBy("farm_id", "sensor_type_key", "reading_date", "date_key").agg(
    F.max("farm_key").alias("farm_key"),
    F.count(F.lit(1)).cast("long").alias("reading_count"),
    F.sum("value").cast("double").alias("sum_value"),
    F.min("value").cast("double").alias("min_value"),
    F.max("value").cast("double").alias("max_value"),
    F.sum("is_anomaly").cast("long").alias("anomaly_count"),
    F.sum(F.lit(1) - F.col("is_anomaly")).cast("long").alias("in_range_count"),
)
target = rollup.select(
    F.col("reading_date").alias("metric_date"), "date_key", "farm_key", "farm_id", "sensor_type_key",
    "reading_count", "sum_value", "min_value", "max_value", "anomaly_count", "in_range_count",
)
write_table(target, "fact_daily_sensor_metrics")

wrote 164700 row(s) -> urbangreen_dw.fact_daily_sensor_metrics


164700

In [18]:
GRAIN = ["farm_id", "metric_date", "date_key"]

harvests = read_query("SELECT farm_id, farm_key, harvest_date, date_key, quality_grade_id, weight_kg FROM fact_harvests FINAL")
recent = harvests
grades = read_query("SELECT quality_grade_id, is_premium FROM dim_quality_grade FINAL")
joined = recent.join(grades, on="quality_grade_id", how="left").withColumn("is_premium", F.coalesce(F.col("is_premium"), F.lit(0)))
premium = F.when(F.col("is_premium") == 1, F.col("weight_kg")).otherwise(0)
non_premium = F.when(F.col("is_premium") == 0, F.col("weight_kg")).otherwise(0)
harvest_side = (
    joined.withColumnRenamed("harvest_date", "metric_date").groupBy(*GRAIN).agg(
        F.max("farm_key").alias("h_farm_key"),
        F.sum("weight_kg").cast("decimal(18,3)").alias("total_yield_kg"),
        F.count(F.lit(1)).cast("int").alias("harvest_count"),
        F.sum(premium).cast("decimal(18,3)").alias("premium_yield_kg"),
        F.sum(non_premium).cast("decimal(18,3)").alias("non_premium_yield_kg"),
    )
)

readings = read_query("SELECT farm_id, farm_key, reading_date, date_key, sensor_type_key, value, is_anomaly, reading_ts FROM fact_sensor_readings FINAL")
recent_r = readings.withColumnRenamed("reading_date", "metric_date")
energy_types = read_query(f"SELECT sensor_type_id AS sensor_type_key FROM dim_sensor_type FINAL WHERE name = '{ENERGY_SENSOR_TYPE}'")
energy_ids = [row["sensor_type_key"] for row in energy_types.collect()]
is_energy = F.col("sensor_type_key").isin(energy_ids) if energy_ids else F.lit(False)
energy_value = F.when(is_energy, F.col("value")).otherwise(0.0)
sensor_side = recent_r.groupBy(*GRAIN).agg(
    F.max("farm_key").alias("s_farm_key"),
    F.count(F.lit(1)).cast("long").alias("reading_count"),
    F.sum("is_anomaly").cast("long").alias("anomaly_count"),
    F.sum(F.lit(1) - F.col("is_anomaly")).cast("long").alias("in_range_count"),
    F.sum(energy_value).cast("double").alias("energy_kwh"),
    F.max("reading_ts").alias("last_sensor_reading_ts"),
)

combined = harvest_side.join(sensor_side, on=GRAIN, how="fullouter")
dates = read_query("SELECT date_key, year_week FROM dim_date")
combined = combined.join(dates, on="date_key", how="left")
target = combined.select(
    "metric_date", "date_key",
    F.coalesce(F.col("h_farm_key"), F.col("s_farm_key"), F.lit(0)).alias("farm_key"), "farm_id",
    F.coalesce(F.col("year_week"), F.lit(0)).alias("year_week"),
    F.coalesce(F.col("total_yield_kg"), F.lit(0)).cast("decimal(18,3)").alias("total_yield_kg"),
    F.coalesce(F.col("harvest_count"), F.lit(0)).cast("int").alias("harvest_count"),
    F.coalesce(F.col("premium_yield_kg"), F.lit(0)).cast("decimal(18,3)").alias("premium_yield_kg"),
    F.coalesce(F.col("non_premium_yield_kg"), F.lit(0)).cast("decimal(18,3)").alias("non_premium_yield_kg"),
    F.coalesce(F.col("energy_kwh"), F.lit(0.0)).cast("double").alias("energy_kwh"),
    F.coalesce(F.col("reading_count"), F.lit(0)).cast("long").alias("reading_count"),
    F.coalesce(F.col("anomaly_count"), F.lit(0)).cast("long").alias("anomaly_count"),
    F.coalesce(F.col("in_range_count"), F.lit(0)).cast("long").alias("in_range_count"),
    F.col("last_sensor_reading_ts"),
)
write_table(target, "fact_daily_farm_metrics")

wrote 63300 row(s) -> urbangreen_dw.fact_daily_farm_metrics


63300

In [19]:
metrics = read_query("SELECT metric_date, date_key, farm_key, farm_id, total_yield_kg, premium_yield_kg, energy_kwh FROM fact_daily_farm_metrics FINAL")
recent = metrics
has_yield = F.col("total_yield_kg") > 0
derived = (
    recent.withColumn("premium_yield_share", F.when(has_yield, F.col("premium_yield_kg") / F.col("total_yield_kg")).otherwise(0.0))
    .withColumn("energy_efficiency_kwh_per_kg", F.when(has_yield, F.col("energy_kwh") / F.col("total_yield_kg")).otherwise(0.0))
    .withColumn("_no_yield", F.when(has_yield, 0).otherwise(1))
)
by_day = Window.partitionBy("date_key")
farm_count = F.count(F.lit(1)).over(by_day)
ranked = (
    derived.withColumn("yield_rank", F.rank().over(by_day.orderBy(F.col("total_yield_kg").desc())))
    .withColumn("quality_rank", F.rank().over(by_day.orderBy(F.col("premium_yield_share").desc())))
    .withColumn("energy_rank", F.rank().over(by_day.orderBy(F.col("_no_yield").asc(), F.col("energy_efficiency_kwh_per_kg").asc())))
    .withColumn("_farm_count", farm_count)
)
points = (
    (F.col("_farm_count") - F.col("yield_rank") + 1)
    + (F.col("_farm_count") - F.col("quality_rank") + 1)
    + (F.col("_farm_count") - F.col("energy_rank") + 1)
)
ranked = ranked.withColumn("composite_score", points.cast("double"))
ranked = ranked.withColumn("composite_rank", F.rank().over(by_day.orderBy(F.col("composite_score").desc())))
target = ranked.select(
    "metric_date", "date_key", "farm_key", "farm_id",
    F.col("total_yield_kg").cast("decimal(18,3)").alias("total_yield_kg"),
    F.col("premium_yield_share").cast("double").alias("premium_yield_share"),
    F.col("energy_efficiency_kwh_per_kg").cast("double").alias("energy_efficiency_kwh_per_kg"),
    F.col("yield_rank").cast("int").alias("yield_rank"),
    F.col("quality_rank").cast("int").alias("quality_rank"),
    F.col("energy_rank").cast("int").alias("energy_rank"),
    "composite_score",
    F.col("composite_rank").cast("int").alias("composite_rank"),
)
write_table(target, "fact_farm_leaderboard")

wrote 63300 row(s) -> urbangreen_dw.fact_farm_leaderboard


63300

In [20]:
read_query("""
SELECT 'dim_role' AS t, count() AS n FROM dim_role FINAL
UNION ALL SELECT 'dim_quality_grade', count() FROM dim_quality_grade FINAL
UNION ALL SELECT 'dim_crop', count() FROM dim_crop FINAL
UNION ALL SELECT 'dim_user', count() FROM dim_user FINAL
UNION ALL SELECT 'dim_sensor_type', count() FROM dim_sensor_type FINAL
UNION ALL SELECT 'dim_farm', count() FROM dim_farm FINAL
UNION ALL SELECT 'dim_sensor', count() FROM dim_sensor FINAL
UNION ALL SELECT 'dim_user_farm_role', count() FROM dim_user_farm_role FINAL
UNION ALL SELECT 'fact_harvests', count() FROM fact_harvests FINAL
UNION ALL SELECT 'fact_sensor_readings', count() FROM fact_sensor_readings FINAL
UNION ALL SELECT 'fact_daily_farm_quality_metrics', count() FROM fact_daily_farm_quality_metrics FINAL
UNION ALL SELECT 'fact_daily_sensor_metrics', count() FROM fact_daily_sensor_metrics FINAL
UNION ALL SELECT 'fact_daily_farm_metrics', count() FROM fact_daily_farm_metrics FINAL
UNION ALL SELECT 'fact_farm_leaderboard', count() FROM fact_farm_leaderboard FINAL
""").orderBy("t").show(30, False)

+-------------------------------+-------+
|t                              |n      |
+-------------------------------+-------+
|dim_crop                       |19     |
|dim_farm                       |75     |
|dim_quality_grade              |5      |
|dim_role                       |3      |
|dim_sensor                     |450    |
|dim_sensor_type                |6      |
|dim_user                       |1      |
|dim_user_farm_role             |1      |
|fact_daily_farm_metrics        |63300  |
|fact_daily_farm_quality_metrics|228037 |
|fact_daily_sensor_metrics      |164700 |
|fact_farm_leaderboard          |63300  |
|fact_harvests                  |1040250|
|fact_sensor_readings           |659700 |
+-------------------------------+-------+

